In [1]:
import pandas as pd
import geopandas as gpd
import json

#show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Clean crosswalk
Ok the main issue we're solving here is that we need to use a crosswalk to get the ONET SOC occupations to match with the BLS (NEM) occupations. But if we take a look at the crosswalk, we realize that there are multiple SOC matches for each NEM. And our AI exposure scores are on the SOC level so when we merge AI exposure scores to BLS employment data by way of the crosswalk, we're going to get dups. How do we know which to choose? Do we take an average?

Our solution is 2-fold. There are 75 NEM codes with more than one SOC match. 
- For the ones that have the option of an exact match between NEM and SOC (49 of the 75), we're going to prioritize that exact match SOC and the study score associated with it
- For the ones that don't have an exact match (26 of the 75) we're going to manually assign the "best" match. How will we do that?
    - Many of these without exact matches are "xxxx, all other" occupations. In our minds, these occupations are just too general to really be able to assign to a specific SOC with a study score. So we're going to use the median or average of all SOC matches
    - The select few that are actually distinct occupations (ex: 31-1120 - Home health and personal care aides), we're going to take a look at the possible options and select the one that looks the best to us.

So basically, this notebook pulls in the official ONET SOC --> NEM crosswalk, and reduces it to a list of unique NEM codes with a single SOC match.

End result: a csv with three columns:
- NEM Code
- SOC Code (can be null for NEMs where we're using the median/avg)
- study score (either from a specific SOC or from median/avg)


### A note on the study score
I also want to make sure everyone know that the study score outputs are adjustable. We've chosen to go with the `human_rating_beta` for this analysis because we thought it was the fairest approach to understanding AI's impact on jobs but feel free to make your own decisions.

- Columns prefixed with `dv_rating_` are how GPT-4 rated the tasks/occupations
- Columns prefixed with `human_rating_` are how the human annotators rated the tasks/occupations
- Columns suffixed with `_alpha` correspond to the share of that occupation's tasks that would directly be made easier by AI. 
- Columns suffixed with `_beta` correspond to the share of that occupation's tasks that would be either directly made easier or would be made easier with additional tools built using AI. A half weight was given to tasks that would be made easier but only with additional AI tools.
- Columns suffixed with `_gamma` is similar to `_beta` but it gives full weight to those tasks made easier only with additional AI tools in the mix.
- According to the study, "made easier" mean that using LLMs via ChatGPT or the OpenAI playground would decrease the time required to complete the task by at least half (50%).

In [2]:
# load data
## study data (has been downloaded to the raw data folder as well)
study_rating = 'human_rating_beta'
outfile = f'../data/processed/study_scores_xwalk_merged_{study_rating}.csv'

occ_scores = pd.read_csv('https://raw.githubusercontent.com/openai/GPTs-are-GPTs/refs/heads/main/data/occ_level.csv')

#adding in the crosswalk
xwalk = pd.read_excel('../data/raw/nem-onet-to-soc-crosswalk.xlsx', sheet_name='ONET to SOC crosswalk', skiprows=4)
xwalk = xwalk[['O*NET-SOC Code','O*NET-SOC Title','NEM Code','National Employment Matrix Occupational Title']]
xwalk = xwalk.rename(columns={'National Employment Matrix Occupational Title': 'NEM Title'})
print('Occ scores pre merge:', len(occ_scores))
occ_scores = occ_scores.merge(xwalk, how='left', on='O*NET-SOC Code')
print('Occ scores post merge w/ xwalk:', len(occ_scores))


occ_scores = occ_scores[['O*NET-SOC Code', 'Title','NEM Code','NEM Title', study_rating]]
print('Total occupation scores:', len(occ_scores))
occ_scores['occupation_code'] = occ_scores['O*NET-SOC Code'].str.replace('-', '')
occ_scores['occupation_code'] = occ_scores['occupation_code'].apply(lambda x: x if len(x) == 6 else x[:6])

occ_scores['nem_code_merge'] = occ_scores['NEM Code'].str.replace('-', '')
occ_scores['nem_code_expanded'] = occ_scores['NEM Code']+'.00'

print('Occupation scores sans detailed:', len(occ_scores))


occ_scores = occ_scores.rename(columns={study_rating: 'study_rating', 
                                        'O*NET-SOC Code': 'soc_code',
                                        'Title': 'occupation_name'})

Occ scores pre merge: 923
Occ scores post merge w/ xwalk: 923
Total occupation scores: 923
Occupation scores sans detailed: 923


In [3]:
def find_exact_matches(df, soc_col):
    """
    This function takes a dataframe and a column name as input and returns the number of exact matches in that column.
    """
    nem_expanded = df['nem_code_merge'].str[:2]+ '-'+ df['nem_code_merge'].str[2:] + '.00'
    return ((df[soc_col] == nem_expanded)).sum()

def calculate_stats(df):
    """
    This function takes a dataframe as input and returns a dictionary with the minimum, maximum, and mean of the 'study_rating' column.
    """
    
    return pd.Series({
        'soc_cnt': df['soc_code'].nunique(),
        'socs': ';'.join(df['soc_plus_score'].unique()),
        'exact_soc_match': find_exact_matches(df, 'soc_code'),
        'study_rating_min': df['study_rating'].min(),
        'study_rating_max': df['study_rating'].max(),
        'study_rating_mean': df['study_rating'].mean()
    })

occ_scores['soc_plus_score'] = occ_scores['occupation_name'] + ' (' + occ_scores['study_rating'].astype(float).round(2).astype(str) + ')'
scores_by_nem = occ_scores.groupby(['NEM Code', 'NEM Title']).apply(calculate_stats).reset_index().sort_values('soc_cnt', ascending=False)

In [4]:
occ_scores.head()

,soc_code,occupation_name,NEM Code,NEM Title,study_rating,occupation_code,nem_code_merge,nem_code_expanded,soc_plus_score
0,11-1011.00,Chief Executives,11-1011,Chief executives,0.350000,111011,111011,11-1011.00,Chief Executives (0.35)
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief executives,0.388889,111011,111011,11-1011.00,Chief Sustainability Officers (0.39)
2,11-1021.00,General and Operations Managers,11-1021,General and operations managers,0.384615,111021,111021,11-1021.00,General and Operations Managers (0.38)
3,11-1031.00,Legislators,11-1031,Legislators,0.516667,111031,111031,11-1031.00,Legislators (0.52)
4,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and promotions managers,0.546512,112011,112011,11-2011.00,Advertising and Promotions Managers (0.55)


In [5]:
print('Total unique NEM codes in the study:', len(scores_by_nem))
print('NEM with more than one SOC match:', len(scores_by_nem.loc[scores_by_nem['soc_cnt'] > 1]))
print('NEM with more than one SOC match but also an exact match:', len(scores_by_nem.loc[(scores_by_nem['soc_cnt'] > 1) & (scores_by_nem['exact_soc_match'] > 0)]))
print('NEM with more than one SOC match but NO exact match:', len(scores_by_nem.loc[(scores_by_nem['soc_cnt'] > 1) & (scores_by_nem['exact_soc_match'] == 0)]))

Total unique NEM codes in the study: 785
NEM with more than one SOC match: 75
NEM with more than one SOC match but also an exact match: 49
NEM with more than one SOC match but NO exact match: 26


In [6]:
#isolate the one-to-one matches
one_to_one_matches = scores_by_nem.loc[scores_by_nem['soc_cnt'] == 1]['NEM Code'].tolist()
scores_one_to_one = occ_scores.loc[occ_scores['NEM Code'].isin(one_to_one_matches)]
scores_one_to_one = scores_one_to_one[['NEM Code','soc_code','study_rating']]

#isolate the multi-match and exact matches
multi_match_exact = scores_by_nem.loc[(scores_by_nem['soc_cnt'] > 1) & (scores_by_nem['exact_soc_match'] > 0)]['NEM Code'].tolist()
scores_multi_exact = occ_scores.loc[(occ_scores['NEM Code'].isin(multi_match_exact))&(occ_scores['nem_code_expanded'] == occ_scores['soc_code'])]
scores_multi_exact = scores_multi_exact[['NEM Code','soc_code','study_rating']]

#isolate the multi-matches with no exact but that have ", all other" in the NEM Title. We'll use the median/avg for those
multi_all_other = scores_by_nem.loc[(scores_by_nem['soc_cnt'] > 1) & (scores_by_nem['exact_soc_match'] == 0) & (scores_by_nem['NEM Title'].str.contains(', all other'))]['NEM Code'].tolist()
scores_multi_all_other = occ_scores.loc[occ_scores['NEM Code'].isin(multi_all_other)]
scores_multi_all_other = scores_multi_all_other.groupby('NEM Code').agg({'study_rating': 'mean'}).reset_index()
scores_multi_all_other['soc_code'] = 'avg from multi-match, all other'

#and pull in our manual matches
manual_matches = pd.read_csv('../data/processed/manual_nem_soc_matches.csv', dtype={'soc_code': str, 'NEM Code': str})
scores_manual = manual_matches.merge(occ_scores[['soc_code','study_rating']], how='left', on='soc_code')

#lastly lets combine and make sure we've got the right number of unique NEM codes
combined_scores = pd.concat([scores_one_to_one, scores_multi_exact, scores_multi_all_other, scores_manual], ignore_index=True)
print('Total unique NEM codes in the combined scores:', len(combined_scores['NEM Code'].unique()))

#jk lastly is the export lol
combined_scores.to_csv(outfile, index=False)

Total unique NEM codes in the combined scores: 785


And then just to make sure we're all good to go, check some stuff.

In [7]:
print('One-to-one matches:', len(scores_one_to_one))
print('Multi-match with exact matches:', len(scores_multi_exact))
print('Multi-match with no exact but "all other":', len(scores_multi_all_other))
print('Manual matches:', len(scores_manual))

One-to-one matches: 710
Multi-match with exact matches: 49
Multi-match with no exact but "all other": 15
Manual matches: 11


In [8]:
785-(710+49+15+11)

0

In [9]:
combined_scores.head()

,NEM Code,soc_code,study_rating
0,11-1021,11-1021.00,0.384615
1,11-1031,11-1031.00,0.516667
2,11-2011,11-2011.00,0.546512
3,11-2021,11-2021.00,0.578125
4,11-2022,11-2022.00,0.483333
